In [ ]:
#!/usr/bin/env python3
"""ISMUN barcode APPENDER — add the extra JCC internal delegates.

This does NOT regenerate the whole set. It reads the existing
ISMUN_BARCODES/id_registry.json, finds the highest number already used,
and appends one new barcode per NAMED person in additional_barcodes.xlsx
(rows with a blank Full name are skipped). New cards are labelled as
Internal Delegate / JCC, exactly like the JCC internal delegates already
in the set.

Run it AFTER the main generator. Safe to run once. If you run it twice it
will detect the same names already present and skip them.

Requires:  pip install python-barcode pillow pandas openpyxl
"""

import os
import re
import json
from pathlib import Path

import pandas as pd
import barcode
from barcode.writer import ImageWriter
from PIL import Image, ImageDraw, ImageFont

DOWNLOADS  = str(Path.home() / "Downloads")
OUTPUT_DIR = os.path.join(DOWNLOADS, "ISMUN_BARCODES")
REGISTRY   = os.path.join(OUTPUT_DIR, "id_registry.json")

COMMITTEE   = "JCC"
GROUP_LABEL = "Internal Delegate"
JPEG_QUALITY = 100
C_ID, C_NAME, C_LINE2 = (2, 132, 199), (15, 23, 42), (71, 85, 105)


def _find_xlsx(*keywords):
    hits = [p for p in Path(DOWNLOADS).glob("*.xlsx")
            if not p.name.startswith("~$")
            and all(k in p.name.lower() for k in keywords)]
    if not hits:
        raise SystemExit(f"No .xlsx in {DOWNLOADS} matching {keywords}")
    hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(hits[0])


def _font(size, bold=False):
    sd = "-Bold" if bold else ""
    sl = "Bold" if bold else "Regular"
    for path in [
        "/Library/Fonts/Arial Bold.ttf" if bold else "/Library/Fonts/Arial.ttf",
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold
        else "/System/Library/Fonts/Supplemental/Arial.ttf",
        f"/usr/share/fonts/truetype/dejavu/DejaVuSans{sd}.ttf",
        f"/usr/share/fonts/truetype/liberation/LiberationSans-{sl}.ttf",
        "C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except (OSError, IOError):
            continue
    try:
        return ImageFont.load_default(size=size)
    except TypeError:
        return ImageFont.load_default()


_FONTS = None


def _fonts():
    global _FONTS
    if _FONTS is None:
        _FONTS = (_font(60, True), _font(68, True), _font(44))
    return _FONTS


def make_barcode_jpg(number, name, line2, path):
    bars = barcode.get("code128", str(number), writer=ImageWriter()).render(
        {"module_width": 0.7, "module_height": 22.0, "quiet_zone": 4.0,
         "write_text": False, "dpi": 600}).convert("RGB")
    f_id, f_name, f_line2 = _fonts()
    pad = 60
    m = ImageDraw.Draw(bars)
    id_txt = f"#{number}"
    id_w = m.textlength(id_txt, font=f_id)
    name_w = m.textlength(name, font=f_name)
    line2_w = m.textlength(line2, font=f_line2)
    width = max(bars.width, int(id_w), int(name_w), int(line2_w)) + pad * 2
    height = bars.height + 360
    canvas = Image.new("RGB", (width, height), "white")
    canvas.paste(bars, ((width - bars.width) // 2, pad))
    d = ImageDraw.Draw(canvas)
    y = pad + bars.height + 24
    d.text(((width - id_w) // 2, y), id_txt, font=f_id, fill=C_ID)
    d.text(((width - name_w) // 2, y + 84), name, font=f_name, fill=C_NAME)
    d.text(((width - line2_w) // 2, y + 176), line2, font=f_line2, fill=C_LINE2)
    canvas.save(path, "JPEG", quality=JPEG_QUALITY, subsampling=0,
                dpi=(600, 600))


def _clean(s):
    return " ".join(str(s).split()).strip()


def main():
    if not os.path.exists(REGISTRY):
        raise SystemExit(f"{REGISTRY} not found — run the main generator "
                         "first.")
    with open(REGISTRY, encoding="utf-8") as f:
        registry = json.load(f)

    highest = max(int(k) for k in registry) if registry else 0
    existing_names = {_clean(v["name"]).casefold() for v in registry.values()}

    src = _find_xlsx("additional", "barcode")
    print("Using:", os.path.basename(src))
    df = pd.read_excel(src)                       # header row 0
    name_c = next((c for c in df.columns
                   if "full name" in str(c).lower()
                   or str(c).lower().strip() == "name"), None)
    if name_c is None:
        raise SystemExit("No 'Full name' column found.")

    to_add = []
    for _, row in df.iterrows():
        raw = row.get(name_c, "")
        if pd.isna(raw):
            continue
        name = _clean(raw)
        if not name or name.lower() in ("nan", "nat"):
            continue
        if name.casefold() in existing_names:
            print(f"  skip (already in set): {name}")
            continue
        to_add.append(name)

    if not to_add:
        print("Nothing new to add.")
        return

    num = highest
    for name in to_add:
        num += 1
        line2 = f"{GROUP_LABEL} | {COMMITTEE}"
        safe = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
        path = os.path.join(OUTPUT_DIR, f"{num:03d}_{safe}.jpg")
        make_barcode_jpg(num, name, line2, path)
        registry[str(num)] = {"name": name, "group": GROUP_LABEL,
                              "committee": COMMITTEE, "position": ""}
        print(f"  added #{num}: {name}")

    with open(REGISTRY, "w", encoding="utf-8") as f:
        json.dump(registry, f, indent=2, ensure_ascii=False)
    print(f"\nDone — added {len(to_add)} barcode(s), "
          f"#{highest + 1}–#{num}. Total now {len(registry)}.")


if __name__ == "__main__":
    main()